In [0]:
from pyspark.sql.functions import col , current_timestamp,regexp_extract, split, date_format , to_date

In [0]:
landing_path    = "/Volumes/skypoints/landing/members"
checkpoint_path = "/Volumes/skypoints/bronze/members/member_checkpoints"


In [0]:
%sql
CREATE TABLE IF NOT EXISTS skypoints.bronze.raw_member
(
    -- fields from the member file, all raw text
    member_id          STRING,
    member_name        STRING,
    country            STRING,
    dob                STRING COMMENT 'Text on purpose: 03051985 must keep its leading zero',
    enrollment_date    STRING,
    last_flight_date   STRING,
    tier               STRING,
    active_flag        STRING,
    agent_name         STRING,
    state              STRING,
    post_code          STRING,

    -- audit
    source_file_name   STRING    COMMENT 'e.g. SKYPOINTS_MEMBERS_20240115_12000000.txt',
    file_date          DATE      COMMENT 'Parsed from the file name',
    ingestion_timestamp TIMESTAMP COMMENT 'When the row was loaded into bronze'
)
USING DELTA
COMMENT 'Bronze: raw member rows, append-only, no casting';

In [0]:

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "text")
    .load(landing_path)
    .select(
        split(col("value"), r"\|").alias("fields"),
        col("_metadata.file_name").alias("source_file_name"),
        to_date(
            regexp_extract(col("_metadata.file_name"), r"_(\d{8})_", 1),
            "yyyyMMdd"
        ).alias("file_date"),
        date_format(
            current_timestamp(), "yyyy-MM-dd HH:mm:ss"
        ).alias("ingestion_timestamp")
    )
    .select(
        col("fields")[1].alias("member_name"),
        col("fields")[2].alias("member_id"),
        col("fields")[3].alias("enrollment_date"),
        col("fields")[4].alias("last_flight_date"),
        col("fields")[5].alias("tier_code"),
        col("fields")[6].alias("agent_name"),
        col("fields")[7].alias("state"),
        col("fields")[8].alias("country"),
        col("fields")[9].alias("post_code"),
        col("fields")[10].alias("dob"),
        col("fields")[11].alias("is_active"),
        col("source_file_name"),
        col("file_date"),
        col("ingestion_timestamp")
    )
)

In [0]:
(
    df.writeStream
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)                   # process what is there, then stop
    .toTable("skypoints.bronze.raw_member")
)

In [0]:
%sql
select * from skypoints.bronze.raw_member;